# RECUPERACIÓN DE LA INFORMACIÓN

# EXAMEN BIMESTRAL

## Nombre: Danny Constante

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset/rotten_tomatoes_movies.csv
/kaggle/input/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset/rotten_tomatoes_critic_reviews.csv


## 0. Instalación de librerías 
   

In [ ]:
# Instalación de dependencias para el Sistema de Recuperación de Información
!pip install sentence-transformers pandas numpy scikit-learn -q

## 1. Carga del Corpus
Se utiliza el dataset *Rotten Tomatoes Movies and Critic Reviews*. Para optimizar el contexto semántico que recibirán los modelos de embeddings, se concatenan el título, los géneros y la crítica en un solo campo textual (`full_text`).

In [39]:
import pandas as pd
import numpy as np

# Rutas obtenidas del entorno de Kaggle
path_movies = '/kaggle/input/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset/rotten_tomatoes_movies.csv'
path_reviews = '/kaggle/input/datasets/stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset/rotten_tomatoes_critic_reviews.csv'

# Cargar los datasets
df_movies = pd.read_csv(path_movies)
df_reviews = pd.read_csv(path_reviews)

print(f"Total de películas: {len(df_movies)}")
print(f"Total de reseñas: {len(df_reviews)}")

# Unir los datasets usando el identificador común ('rotten_tomatoes_link')
df_corpus = pd.merge(df_reviews, df_movies, on='rotten_tomatoes_link', how='inner')

# Selección de campos textuales requeridos
df_corpus = df_corpus[['rotten_tomatoes_link', 'movie_title', 'review_content', 'genres']]

# Limpiar valores nulos para evitar errores en los embeddings
df_corpus = df_corpus.dropna(subset=['review_content', 'movie_title']).reset_index(drop=True)

# MUESTREO (Opcional pero fuertemente recomendado en Kaggle para evitar TimeOuts en el examen)
# Trabajaremos con 30,000 registros para mantener la ejecución fluida.
df_corpus = df_corpus.sample(30000, random_state=42).reset_index(drop=True)

# Consolidar el documento textual para los Sentence Transformers
df_corpus['full_text'] = df_corpus['movie_title'] + " " + df_corpus['genres'].fillna('') + " " + df_corpus['review_content']

# Renombrar columna de ID
df_corpus = df_corpus.rename(columns={'rotten_tomatoes_link': 'doc_id'})

display(df_corpus[['doc_id', 'full_text']].head(3))

Total de películas: 17712
Total de reseñas: 1130017


,doc_id,full_text
0,m/reality_2015,Reality (Réalité) Drama [A] mind-numbingly unf...
1,m/tenderness_of_the_wolves,Die Zärtlichkeit der Wölfe (The Tenderness of ...
2,m/jackass_3,"Jackass 3 Action & Adventure, Comedy, Document..."


## 2. Preprocesamiento Aplicado
Se implementa un pipeline que cumple con los requisitos obligatorios:
* Conversión a minúsculas (*Case folding*).
* Eliminación de signos de puntuación.
* Eliminación de espacios redundantes.

**Decisión de diseño:** Para mantener el rigor en la Recuperación de Información, no se aplica *stemming* agresivo ni eliminación masiva de *stopwords*. Los modelos de representaciones densas (Embeddings) necesitan preposiciones y conectores para inferir la semántica y relaciones en las oraciones. Eliminar estas palabras restaría precisión al contexto.

In [40]:
import re

def preprocess_text(text):

    # 1. Conversión a minúsculas
    text = str(text).lower()
    
    # 2. Eliminación de signos de puntuación (mantiene letras y números)
    # Se reemplaza cualquier caracter que no sea alfanumérico o espacio por un espacio
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 3. Eliminación de espacios redundantes
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Aplicar el preprocesamiento al corpus
df_corpus['processed_text'] = df_corpus['full_text'].apply(preprocess_text)

print("\nComparación antes y después del preprocesamiento:")
display(df_corpus[['full_text', 'processed_text']].head(3))


Comparación antes y después del preprocesamiento:


,full_text,processed_text
0,Reality (Réalité) Drama [A] mind-numbingly unf...,reality réalité drama a mind numbingly unfunny...
1,Die Zärtlichkeit der Wölfe (The Tenderness of ...,die zärtlichkeit der wölfe the tenderness of w...
2,"Jackass 3 Action & Adventure, Comedy, Document...",jackass 3 action adventure comedy documentary ...


## 3. Generación de Embeddings 

Para cumplir con los requisitos del Nivel Sobresaliente, se implementa una comparación entre dos modelos de embeddings de la librería `sentence-transformers`:

1. **`all-MiniLM-L6-v2`**: Modelo ligero y rápido (384 dimensiones).
2. **`all-mpnet-base-v2`**: Modelo de mayor densidad y capacidad semántica (768 dimensiones).

Se codifica el documento completo (`processed_text`) en lugar de palabras sueltas, permitiendo que la arquitectura Transformer capture el significado global de la reseña junto al género y título de la película.

In [41]:
from sentence_transformers import SentenceTransformer
import torch

# Verificamos si hay una GPU disponible para acelerar el proceso
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de procesamiento detectado: {device}\n")

# Inicializamos ambos modelos
print("Cargando modelo 1: all-MiniLM-L6-v2...")
model_minilm = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print("Cargando modelo 2: all-mpnet-base-v2...")
model_mpnet = SentenceTransformer('all-mpnet-base-v2', device=device)

# Extraemos los textos preprocesados a una lista
textos_corpus = df_corpus['processed_text'].tolist()

# Generación de Embeddings
print("\nGenerando embeddings con all-MiniLM-L6-v2 (Modelo Ligero)...")
# Usamos show_progress_bar para monitorear el avance
embeddings_minilm = model_minilm.encode(textos_corpus, batch_size=64, show_progress_bar=True)

print("\nGenerando embeddings con all-mpnet-base-v2 (Modelo Denso)...")
embeddings_mpnet = model_mpnet.encode(textos_corpus, batch_size=64, show_progress_bar=True)

print("\n¡Embeddings generados exitosamente!")
print(f"Dimensiones del corpus MiniLM: {embeddings_minilm.shape} (Documentos x Dimensiones)")
print(f"Dimensiones del corpus MPNet: {embeddings_mpnet.shape} (Documentos x Dimensiones)")

Dispositivo de procesamiento detectado: cuda

Cargando modelo 1: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cargando modelo 2: all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...


Generando embeddings con all-MiniLM-L6-v2 (Modelo Ligero)...


Batches:   0%|          | 0/469 [00:00<?, ?it/s]


Generando embeddings con all-mpnet-base-v2 (Modelo Denso)...


Batches:   0%|          | 0/469 [00:00<?, ?it/s]


¡Embeddings generados exitosamente!
Dimensiones del corpus MiniLM: (30000, 384) (Documentos x Dimensiones)
Dimensiones del corpus MPNet: (30000, 768) (Documentos x Dimensiones)


## 4. Procesamiento de Consultas y Recuperación

Se implementa un mecanismo de recuperación utilizando la **Similitud Coseno** para comparar el vector de la consulta con todos los vectores de los documentos. 

Para cumplir con el desafío de excelencia, la función de búsqueda permite seleccionar qué conjunto de embeddings utilizar (`all-MiniLM-L6-v2` o `all-mpnet-base-v2`), devolviendo un DataFrame formateado con el Top-k de resultados que incluye el Ranking, ID del documento, Título, Fragmento y Similitud.

In [55]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar_documentos(query, df, corpus_embeddings, model, k=5):
    """
    Recibe una consulta, genera su embedding, calcula la similitud coseno contra el corpus
    y devuelve los k documentos más relevantes.
    """
    # 1. Preprocesar la consulta de la misma forma que el corpus
    processed_query = preprocess_text(query)
    
    # 2. Generar el embedding de la consulta 
    query_embedding = model.encode([processed_query])
    
    # 3. Calcular la similitud coseno 
    # cosine_similarity devuelve una matriz, tomamos la primera fila [0]
    similitudes = cosine_similarity(query_embedding, corpus_embeddings)[0]
    
    # 4. Obtener los índices de los k documentos con mayor similitud
    # argsort ordena de menor a mayor, tomamos los últimos k y los invertimos [::-1]
    top_k_indices = similitudes.argsort()[-k:][::-1]
    
    # 5. Construir la tabla de resultados solicitada 
    resultados = []
    for rank, idx in enumerate(top_k_indices, start=1):
        resultados.append({
            'Ranking': rank,
            'ID documento': df.iloc[idx]['doc_id'],
            'Título película': df.iloc[idx]['movie_title'],
            'Fragmento de texto': df.iloc[idx]['full_text'][:150] + "...", # Mostramos un fragmento
            'Similitud': round(similitudes[idx], 4)
        })
        
    return pd.DataFrame(resultados)

## 5. Benchmark de Consultas Obligatorias

A continuación, se ejecutan las 8 consultas requeridas. Se procesa una de las consultas con ambos modelos para evidenciar la diferencia en la recuperación, y luego construiremos la tabla resumen general

In [56]:
# Lista de consultas obligatorias
consultas = [
    "science fiction movie with advanced technology", # Q1
    "romantic story with emotional relationships",    # Q2 
    "action movie with intense fight scenes",         # Q3 
    "horror film that creates fear and suspense",     # Q4 
    "visually impressive movie with weak storyline",  # Q5 
    "emotionally moving performance by the lead actor",# Q6 
    "predictable plot but entertaining experience",   # Q7 
    "movie praised by critics but unpopular with audiences" # Q8 
]

# Diccionario para guardar el mejor resultado de cada consulta para la tabla resumen
resumen_general = []

print("=== RESULTADOS UTILIZANDO EL MODELO DENSO (all-mpnet-base-v2) ===")

for i, query in enumerate(consultas, start=1):
    print(f"\nConsulta Q{i}: '{query}'")
    
    # Búsqueda usando el modelo MPNet
    df_resultados = buscar_documentos(query, df_corpus, embeddings_mpnet, model_mpnet, k=5)
    
    # Mostrar la tabla del Top-K para esta consulta 
    display(df_resultados)
    
    # Guardar el Top-1 para la tabla resumen 
    top_1 = df_resultados.iloc[0]
    resumen_general.append({
        'Consulta': f"Q{i}",
        'Query Text': query,
        'Documento Top-1 (ID)': top_1['ID documento'],
        'Título película': top_1['Título película'],
        'Similitud': top_1['Similitud']
    })

# Construcción de la Tabla Resumen General 
df_resumen = pd.DataFrame(resumen_general)
print("\n=== TABLA RESUMEN GENERAL (MEJOR RESULTADO POR CONSULTA) ===")
display(df_resumen)

=== RESULTADOS UTILIZANDO EL MODELO DENSO (all-mpnet-base-v2) ===

Consulta Q1: 'science fiction movie with advanced technology'


,Ranking,ID documento,Título película,Fragmento de texto,Similitud
0,1,m/interstellar_2014,Interstellar,"Interstellar Action & Adventure, Science Ficti...",0.6169
1,2,m/terminator,The Terminator,"The Terminator Action & Adventure, Science Fic...",0.5935
2,3,m/kin_2018,Kin,"Kin Action & Adventure, Science Fiction & Fant...",0.5921
3,4,m/i_robot,"I, Robot","I, Robot Action & Adventure, Mystery & Suspens...",0.5910
4,5,m/interstellar_2014,Interstellar,"Interstellar Action & Adventure, Science Ficti...",0.5826



Consulta Q2: 'romantic story with emotional relationships'


,Ranking,ID documento,Título película,Fragmento de texto,Similitud
0,1,m/marriage_story_2019,Marriage Story,Marriage Story Drama [A]n exquisite heartbreak...,0.6759
1,2,m/marriage_story_2019,Marriage Story,Marriage Story Drama Marriage Story is a treme...,0.6479
2,3,m/two_lovers,Two Lovers,"Two Lovers Drama, Romance We know the characte...",0.6308
3,4,m/carol,Carol,"Carol Drama, Romance A very intelligent and so...",0.6262
4,5,m/1104841-sweet_november,Sweet November,"Sweet November Drama, Romance Starts off sappy...",0.6180



Consulta Q3: 'action movie with intense fight scenes'


...

,Ranking,ID documento,Título película,Fragmento de texto,Similitud
0,1,m/1133499-1133499-terminal,The Terminal,"The Terminal Comedy, Drama Much of the action ...",0.6688
1,2,m/we_own_the_night,We Own the Night,"We Own the Night Action & Adventure, Drama Thi...",0.6680
2,3,m/no_reservations,No Reservations,"No Reservations Comedy, Drama, Romance Inconse...",0.6608
3,4,m/1116086-trapped,Trapped,"Trapped Action & Adventure, Drama, Mystery & S...",0.6557
4,5,m/serendipity,Serendipity,"Serendipity Comedy, Romance Superficial and pr...",0.6378



Consulta Q8: 'movie praised by critics but unpopular with audiences'


,Ranking,ID documento,Título película,Fragmento de texto,Similitud
0,1,m/ripd,R.I.P.D.,"R.I.P.D. Action & Adventure, Comedy It's not f...",0.6891
1,2,m/win_win_2011,Win Win,"Win Win Comedy, Drama For all its clichs, it w...",0.6045
2,3,m/sex_lies_and_videotape,"Sex, Lies, and Videotape","Sex, Lies, and Videotape Art House & Internati...",0.5956
3,4,m/xxx_state_of_the_union,xXx: State of the Union,xXx: State of the Union Action & Adventure One...,0.5954
4,5,m/because_i_said_so,Because I Said So,"Because I Said So Comedy, Drama, Romance each ...",0.5751



=== TABLA RESUMEN GENERAL (MEJOR RESULTADO POR CONSULTA) ===


,Consulta,Query Text,Documento Top-1 (ID),Título película,Similitud
0,Q1,science fiction movie with advanced technology,m/interstellar_2014,Interstellar,0.6169
1,Q2,romantic story with emotional relationships,m/marriage_story_2019,Marriage Story,0.6759
2,Q3,action movie with intense fight scenes,m/never_back_down,Never Back Down,0.6922
3,Q4,horror film that creates fear and suspense,m/sinister_2012,Sinister,0.7439
4,Q5,visually impressive movie with weak storyline,m/million_dollar_arm,Million Dollar Arm,0.6880
5,Q6,emotionally moving performance by the lead actor,m/a_better_life,A Better Life,0.6042
6,Q7,predictable plot but entertaining experience,m/1133499-1133499-terminal,The Terminal,0.6688
7,Q8,movie praised by critics but unpopular with au...,m/ripd,R.I.P.D.,0.6891
